<a href="https://colab.research.google.com/github/Tracykeku/PaySim-Fraud-Detection/blob/main/FRAUD_DETECTION_MODEL_USING_PAYSIM_DATASET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Track: Data Scientist (DS-1,PaySim fraud)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/PS_20174392719_1491204439457_log.csv')

Mounted at /content/drive


The approach used here is CRISP-DM.

1. Business Understanding

The goal isn't to build the most accurate model but to help a team catch a significant share of real fraud without being flooded with false alarms.

Assumption - This is synthetic data, scaled down from real logs of mobile money. The dataset documentation also warns that several balance columns leak the outcome,since fraud transactions are cancelled in the simulation.This shaped the choices made
from the start.

2. Data Understanding

The dataset is loaded and two things are checked:

a. How imbalanced is the fraud

b. If fraud happens across all transaction types or only some.

In [5]:
import pandas as pd
print(df.shape)#to check total number of rows and columns
print(df.dtypes)#to check the datatypes of every column
print(df.head())# to show the first five rows
print(df.tail())# to show the last rows
# Checking for class imbalance
print(df['isFraud'].value_counts()) #counts how many times each unique value appears in a column
print(df['isFraud'].value_counts(normalize=True))# converts it to a proportion in form of percentage

# Checking if fraud happens across all transaction types
print(df.groupby('type')['isFraud'].sum())#splits data into groups,based on the type
print(df['type'].value_counts())#converts it to a proportion

(6362620, 11)
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C55326406

Fraud is only 0.129% of all transactions. 6,354,407 transactions are legitimate while  8,213 transactions are fraud.

Fraud occurs only in TRANSFER and CASH_OUT transactions.There are zero fraud cases in
CASH_IN, DEBIT, or PAYMENT, across millions of rows.

3. Problem Scope

Since fraud cannot occur outside TRANSFER or CASH_OUT transactions,the dataset was filtered to
just those two types.

In [6]:
df_scoped = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
print(df_scoped.shape)
print(df_scoped['isFraud'].value_counts(normalize=True))

(2770409, 11)
isFraud
0    0.997035
1    0.002965
Name: proportion, dtype: float64


2,770,409 rows remain and fraud rate rises from 0.129% to 0.297%.

4. Feature judgement.

This was done by going column by column, assessing if the feature is available at prediction time, and if it is fair to use.

Step, type and amount are retained as they are known before the transaction is decided.

oldbalanceOrg, oldbalanceDest are retained as they are known before the transaction and are verified to be genuine signals and not leakage.

newbalanceOrig and newbalanceDest are dropped since these leak the outcome.Fraudulent transactions get cancelled in the simulation, meaning these balance columns don't reflect trustworthy outcomes for fraud cases. Therefore,this creates a target leak.Using this columns results to perfect models because the balance columns are quietly giving away the answer.

nameOrig and nameDest are dropped since they are all unique IDs  and have no generalizable behavioral signal.

isFlaggedFraud is dropped since it is an output of an existing rule based flag where transfer amounts greater than 200,000 are flagged as fraud.

In [7]:
print(df_scoped.groupby('isFraud')[['amount','oldbalanceOrg','newbalanceOrig']].describe())

            amount                                                            \
             count          mean           std   min          25%        50%   
isFraud                                                                        
0        2762196.0  3.141155e+05  8.771441e+05  0.01   82908.2325  171034.46   
1           8213.0  1.467967e+06  2.404253e+06  0.00  127091.3300  441423.44   

                                  oldbalanceOrg                ...  \
                 75%          max         count          mean  ...   
isFraud                                                        ...   
0         305994.185  92445516.64     2762196.0  4.287969e+04  ...   
1        1517771.480  10000000.00        8213.0  1.649668e+06  ...   

                                 newbalanceOrig                               \
                75%          max          count           mean           std   
isFraud                                                                        
0       

5. Train/Test split

A stratified 80/20 split was used rather than a plain random split, since fraud is rare in the dataset.

Stratifying keeps the fraud ratio consistent across both sets.


In [8]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
features = ['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
X = df_scoped[features].copy()
y = df_scoped['isFraud'].copy()

# Turns'type' to numeric 0 or 1 since it's categorical (TRANSFER vs CASH_OUT)
X = pd.get_dummies(X, columns=['type'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,#20% for testing
    random_state=42,#makes the code reproducible
    stratify=y  # to keep fraud ratio consistent across both sets
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(2216327, 5) (554082, 5)
isFraud
0    0.997036
1    0.002964
Name: proportion, dtype: float64
isFraud
0    0.997035
1    0.002965
Name: proportion, dtype: float64


6. Model

Logistic regression with class_weight= balanced is chosen as a first model because its coefficients show direction and strength of the effect of each feature.
It is also a legitimate baseline before reaching for more complex models.

class_weight= balanced forces the model to learn the rare class instead of defaulting to always predict non fraud.

In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [10]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # probability of fraud.

7. Evaluation

Accuracy is meaningless here, a model predicting never fraud would already be 99.7% accurate.
Metrics such as recall,precision,confusion matrix,ROC_AUC ,F1 Score and PR-AUC are used.

In [11]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# precision, recall, f1 for both classes
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# metrics focused on the fraud class
print("\nPrecision (fraud class):", precision_score(y_test, y_pred))
print("Recall (fraud class):", recall_score(y_test, y_pred))
print("F1 (fraud class):", f1_score(y_test, y_pred))

# ROC-AUC and PR-AUC
print("\nROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC (average precision):", average_precision_score(y_test, y_pred_proba))

Confusion Matrix:
[[507061  45378]
 [   248   1395]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9995    0.9179    0.9569    552439
           1     0.0298    0.8491    0.0576      1643

    accuracy                         0.9177    554082
   macro avg     0.5147    0.8835    0.5073    554082
weighted avg     0.9966    0.9177    0.9543    554082


Precision (fraud class): 0.029824898980180873
Recall (fraud class): 0.8490566037735849
F1 (fraud class): 0.05762557832121613

ROC-AUC: 0.9516530687165966
PR-AUC (average precision): 0.20075777837793143


True Positives (1,395): fraud cases correctly caught

False Negatives (248): real fraud missed

False Positives (45,378): legitimate transactions wrongly flagged as fraud

True Negatives (507,061): legitimate transactions correctly left alone

Recall ( 84.9%) - Of all 1,643 real fraud cases in the test set, 1,395 of them were caught.

Precision ( 2.98%)  — Of everything flagged as fraud (46,773 transactions total), only 1,395 were actually fraud.

ROC-AUC (0.95) is measured against the huge non-fraud class, so it stays high even with many false positives.

PR-AUC (0.20) on the other hand measures precision directly on the positive class, which is a much stricter test given only a small percentage of the transactions are fraud.

8. Threshold Selection and Business Translation

Several thresholds were tested to see the precision/recall
trade off , then chosen based on the real cost of each mistake.A missed fraud case is more costly than a false alarm.



In [17]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

thresholds = [0.3, 0.5, 0.7, 0.9, 0.95]

for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred_t)
    rec = recall_score(y_test, y_pred_t)
    flagged = y_pred_t.sum()
    print(f"Threshold {t}: Precision={prec:.4f}, Recall={rec:.4f}, Total flagged={flagged}")

Threshold 0.3: Precision=0.0169, Recall=0.9197, Total flagged=89532
Threshold 0.5: Precision=0.0298, Recall=0.8491, Total flagged=46773
Threshold 0.7: Precision=0.0455, Recall=0.7310, Total flagged=26391
Threshold 0.9: Precision=0.0686, Recall=0.5721, Total flagged=13708
Threshold 0.95: Precision=0.0799, Recall=0.4973, Total flagged=10223


The chosen threshold was 0.5 since it catches 85% of real fraud, flagging 46,773 of 554,082 transactions for review,meaning a review team would need to check about 8.4% of all transactions, of which only about 3% turn out to be actual fraud.

A stricter threshold would cut down the volume of transactions to be reviewed but let nearly half of fraud go undetected which is too risky.